# Import 

In [1]:
# Load classification functions
from classify_reactions_nb_03 import *

import sys
sys.path.append('../../../')
sys.path.append('../../')

from file_management import get_files_dir, check_save_file

import pandas as pd

# Base directories
_, INPUT_DIR, OUTPUT_DIR = get_files_dir()

# Files

## Input

In [2]:
# Full text with extracted gene information
INPUT_FILE = OUTPUT_DIR + "/Articles/full_text_w_genes_25_09_26_new_v.json"
full_text_df = pd.read_json(INPUT_FILE)

full_text_df.head()

,Title,Abstract,Journal,Year,PMC_ID,DOI,Type,Author,Text,Abstract_min,Title_min,Full_text_min,Old_name,Organism_Source,Organism,Genes,Gene_Source
10618204,Increased production of zeaxanthin and other p...,The psbAII locus was used as an integration pl...,Applied and environmental microbiology,2000,91786.0,10.1128/AEM.66.1.64-72.2000,"Journal Article,Research Support, U.S. Gov't, ...","[ForeName:D,LastName:Lagarde] [ForeName:L,Last...",None,psbAII overexpress Synechocystis . PCC psbAII ...,pigments techniques Synechocystis . PCC .,None,[Synechocystis PCC],title,[Synechocystis PCC],"[[crtB, P37294, syn:slr1255;, 2.5.1.32;], [crt...",abstract
10618209,Expression of Alcaligenes eutrophus flavohemop...,Expression of the vhb gene encoding hemoglobin...,Applied and environmental microbiology,2000,91791.0,10.1128/AEM.66.1.98-104.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:A D,LastName:Frey] [ForeName:J E,Las...",None,vhb Vitreoscilla . (VHb) organisms microaerobi...,Alcaligenes eutrophus flavohemoprotein Vitreos...,None,[Escherichia coli],title,[Escherichia coli],None,not_found
10631776,Environmental biotechnology.,There is an increasing interest in environment...,Trends in biotechnology,2000,NaN,10.1016/s0167-7799(99)01399-2,"Journal Article,","[ForeName:L P,LastName:Wackett]",None,"world' maintain soil, water. biology. Plants r...",Environmental biotechnology.,None,None,not_found,None,None,not_found
10649237,Altered regulation of pyruvate kinase or co-ov...,Glycolytic fluxes in resting Escherichia coli ...,Biotechnology and bioengineering,2000,NaN,10.1002/(sici)1097-0290(20000305)67:5<623::aid...,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:M,LastName:Emmerling] [ForeName:J E,...",None,Glycolytic resting kinases (Pyk) stearothermop...,Altered co- phosphofructokinase glycolytic res...,None,[Escherichia coli],title,[Escherichia coli],None,not_found
10649449,Cloning and characterization of the Yarrowia l...,The squalene synthase (SQS) gene encodes a key...,"Yeast (Chichester, England)",2000,NaN,10.1002/(SICI)1097-0061(200002)16:3<197::AID-Y...,"Journal Article,Research Support, Non-U.S. Gov...","[ForeName:S,LastName:Merkulov] [ForeName:F,Las...",None,"squalene (SQS) encodes , farnesyl-diphosphate ...",Cloning lipolytica squalene (SQS1) erg9 .,None,"[Yarrowia lipolytica, Saccharomyces cerevisiae]",title,"[Saccharomyces cerevisiae, Yarrowia lipolytica]","[[SQS1, A6ZRL6_P53866A6ZRL6_P53866, sce:YNL224...",title


In [3]:
# Load the normalized EC modifications produced earlier
classified_input = "Test_normalized_v4.json"
merged_df = pd.read_json(classified_input)

merged_df.head()

,Title,Organism,Genes,Gene_Source,Modifications,EC_Modifications,EC_Modifications_Normalized
0,Increased production of zeaxanthin and other p...,[Synechocystis PCC],"[[crtB, P37294, syn:slr1255;, 2.5.1.32;], [crt...",abstract,"[ipi -> expression, crtP -> was introduced, cr...","{'ipi': {'EC_numbers': [], 'Modifications': ['...","{'ipi': {'EC_numbers': [], 'Normalized_Modific..."
1,Expression of Alcaligenes eutrophus flavohemop...,[Escherichia coli],None,not_found,None,{},{}
2,Environmental biotechnology.,None,None,not_found,None,{},{}
3,Altered regulation of pyruvate kinase or co-ov...,[Escherichia coli],None,not_found,[pyk -> overexpression],"{'pyk': {'EC_numbers': ['2.7.1.40'], 'Modifica...","{'pyk': {'EC_numbers': ['2.7.1.40'], 'Normaliz..."
4,Cloning and characterization of the Yarrowia l...,"[Saccharomyces cerevisiae, Yarrowia lipolytica]","[[SQS1, A6ZRL6_P53866A6ZRL6_P53866, sce:YNL224...",title,None,{},{}


## Output

# Classify EC Modifications

In [ ]:
# 1) First full classification
merged_df["EC_Modifications_Classified"] = (
    merged_df["EC_Modifications_Normalized"].apply(classify_mod_dict)
)

merged_df.head()

In [5]:
merged_df.EC_Modifications_Classified.dropna().iloc[0]

{'ipi': {'EC_numbers': [],
  'Classified_Modifications': [{'mod': 'express', 'class': 'Positive'},
   {'mod': 'overexpress', 'class': 'Positive'},
   {'mod': 'introduc', 'class': 'Positive'}]},
 'crtP': {'EC_numbers': [],
  'Classified_Modifications': [{'mod': 'introduc', 'class': 'Positive'},
   {'mod': 'express', 'class': 'Positive'},
   {'mod': 'overexpress', 'class': 'Positive'}]},
 'crtB': {'EC_numbers': [],
  'Classified_Modifications': [{'mod': 'introduc', 'class': 'Positive'},
   {'mod': 'overexpress', 'class': 'Positive'},
   {'mod': 'express', 'class': 'Positive'}]},
 'crtR': {'EC_numbers': [],
  'Classified_Modifications': [{'mod': 'overexpress', 'class': 'Positive'},
   {'mod': 'introduc', 'class': 'Positive'},
   {'mod': 'express', 'class': 'Positive'}]}}

In [6]:
# 2) Extract only the relevant classification subset
merged_df["EC_Modifications_Classified_min"] = (
    merged_df["EC_Modifications_Classified"].apply(extract_relevant_dict)
)

# 3) Convert to complete EC number sets
merged_df["EC_Modifications_Classified_min_relv"] = (
    merged_df["EC_Modifications_Classified_min"].apply(extract_complete_EC_numbers_dict)
)

# 4) Convert empty lists to None for clean merging later
merged_df["EC_Modifications_Classified_min_relv"] = (
    merged_df["EC_Modifications_Classified_min_relv"].apply(lambda x: None if x == [] else x)
)

merged_df.head(10)

,Title,Organism,Genes,Gene_Source,Modifications,EC_Modifications,EC_Modifications_Normalized,EC_Modifications_Classified,EC_Modifications_Classified_min,EC_Modifications_Classified_min_relv
0,Increased production of zeaxanthin and other p...,[Synechocystis PCC],"[[crtB, P37294, syn:slr1255;, 2.5.1.32;], [crt...",abstract,"[ipi -> expression, crtP -> was introduced, cr...","{'ipi': {'EC_numbers': [], 'Modifications': ['...","{'ipi': {'EC_numbers': [], 'Normalized_Modific...","{'ipi': {'EC_numbers': [], 'Classified_Modific...",None,None
1,Expression of Alcaligenes eutrophus flavohemop...,[Escherichia coli],None,not_found,None,{},{},{},None,None
2,Environmental biotechnology.,None,None,not_found,None,{},{},{},None,None
3,Altered regulation of pyruvate kinase or co-ov...,[Escherichia coli],None,not_found,[pyk -> overexpression],"{'pyk': {'EC_numbers': ['2.7.1.40'], 'Modifica...","{'pyk': {'EC_numbers': ['2.7.1.40'], 'Normaliz...","{'pyk': {'EC_numbers': ['2.7.1.40'], 'Classifi...","{'pyk': {'EC_numbers': ['2.7.1.40'], 'Classifi...","{'pyk': {'EC_numbers': ['2.7.1.40'], 'Classifi..."
4,Cloning and characterization of the Yarrowia l...,"[Saccharomyces cerevisiae, Yarrowia lipolytica]","[[SQS1, A6ZRL6_P53866A6ZRL6_P53866, sce:YNL224...",title,None,{},{},{},None,None
5,A novel genetically engineered pathway for syn...,[Escherichia coli],"[[phaC, A0A3L0W3F5, 0, 0]]",abstract,"[phaC -> expressing, buk -> expressing, ptb ->...","{'phaC': {'EC_numbers': ['2.3.1.-'], 'Modifica...","{'phaC': {'EC_numbers': ['2.3.1.-'], 'Normaliz...","{'phaC': {'EC_numbers': ['2.3.1.-'], 'Classifi...","{'phaC': {'EC_numbers': ['2.3.1.-'], 'Classifi...","{'buk': {'EC_numbers': ['2.7.2.7'], 'Classifie..."
6,Formation of functional heterologous complexes...,"[Saccharopolyspora erythraea, Escherichia coli]","[[eryAI, A4F7N8_O33937_Q5UNP6, sen:SACE_0721;,...",full_text,"[pikAIII -> removed, PKS -> express, PKS -> en...","{'pikAIII': {'EC_numbers': [], 'Modifications'...","{'pikAIII': {'EC_numbers': [], 'Normalized_Mod...","{'pikAIII': {'EC_numbers': [], 'Classified_Mod...",None,None
7,Metabolic engineering of Alcaligenes eutrophus...,None,None,not_found,None,{},{},{},None,None
8,Fermentation process kinetics. Reprinted from ...,None,None,not_found,None,{},{},{},None,None
9,Protein engineering of cytochrome p450(cam) (C...,[Pseudomonas putida],None,not_found,None,{},{},{},None,None


In [11]:
exploded_df = explode_ec_modifications_with_paper_id(merged_df)
exploded_df.head(6)

,Paper_ID,Gene,EC_number,Modification,Classification
0,3,pyk,[2.7.1.40],overexpress,Positive
1,5,buk,[2.7.2.7],express,Positive
2,5,ptb,[2.3.1.19],express,Positive
3,5,phaE,"[6.2.1.30, 4.2.1.17]",express,Positive
4,10,phbB,[1.1.1.36],introduct,Positive
5,15,dxs,[2.2.1.7],overexpress,Positive


In [12]:
exploded_df = exploded_df.loc[:,['Paper_ID', 'Gene', 'EC_number', 'Classification']]

In [13]:
repeated = exploded_df.loc[:,['Paper_ID','Gene','Classification']].drop_duplicates().index
repeated.shape

lista = exploded_df.loc[repeated,['Paper_ID','Gene','EC_number','Classification']].copy()

In [17]:
lista

,Paper_ID,Gene,EC_number,Classification
0,3,pyk,[2.7.1.40],Positive
1,5,buk,[2.7.2.7],Positive
2,5,ptb,[2.3.1.19],Positive
3,5,phaE,"[6.2.1.30, 4.2.1.17]",Positive
4,10,phbB,[1.1.1.36],Positive
...,...,...,...,...
19909,16117,PDC1,[4.1.1.1],Positive
19910,16118,sacA,[3.2.1.26],Positive
19911,16118,kfiD,[1.1.1.22],Positive
19912,16118,galU,[2.7.7.9],Positive


In [18]:
# Keep original form for later merging
original_df = merged_df.copy()

#Genes with the classified modification
lista.to_json('Gene_modification_Clasification.json')

# Save classified merged dataset
original_df.to_json("Test_normalised_classifed.json")